# Phase 1 — Baseline Qwen3.5-9B trên 4 datasets × {zero-shot, few-shot, symbolic-cot}

Notebook này tổ chức mỗi experiment trong **1 cell riêng** để dễ debug: khi 1 experiment lỗi/lạ, bạn chỉ cần chỉnh & rerun cell đó — không ảnh hưởng các experiment khác.

Model Qwen3.5-9B chỉ load **một lần** ở cell setup rồi tái sử dụng cho cả 12 experiment → tiết kiệm VRAM load time.

Mỗi cell experiment bật **verbose** để in:
- Full log 5 sample đầu (câu hỏi, raw output, extracted, gold, correct, thời gian).
- Log rút gọn mỗi N sample (xem `verbose_every`).
- Running metric (F1 / accuracy) mỗi 100 sample để phát hiện sớm prompt kém hoặc extractor sai.

**Symbolic CoT** (EXP 9–12): LLM sinh Python datetime program → symbolic executor chạy deterministic → verify kết quả → self-correct nếu sai. Cấu hình qua `n_hypotheses` và `max_correction_attempts` trong `run_exp()` hoặc trực tiếp trong YAML config.

## Setup (chạy tuần tự 1 lần)

In [ ]:
# === SETUP 1 — cài môi trường ===
# Ollama handles model serving locally — no transformers/GPU packages needed.
!pip install -q -U scikit-learn pyyaml pandas

In [ ]:
# === SETUP 2 — mount Drive + clone repo ===
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/<YOUR_USER>/Temporal_Reasoning.git'  # TODO: đổi
REPO_DIR = '/content/Temporal_Reasoning'
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
os.chdir(REPO_DIR)
print('CWD:', os.getcwd())

In [ ]:
# === SETUP 3 — cấu hình path + link Dataset từ Drive ===
import os, sys

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _in_colab():
    DRIVE_OUT     = '/content/drive/MyDrive/Temporal_Reasoning/outputs'
    DATASET_ROOT  = '/content/drive/MyDrive/Temporal_Reasoning/Dataset'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    local_ds = os.path.join(REPO_DIR, 'Dataset')
    if not os.path.exists(local_ds):
        os.symlink(DATASET_ROOT, local_ds)
    print('Dataset ->', os.readlink(local_ds) if os.path.islink(local_ds) else local_ds)
else:
    # Local execution — paths relative to repo root
    _repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
    DRIVE_OUT    = os.path.join(_repo_root, 'outputs')
    DATASET_ROOT = os.path.join(_repo_root, 'Dataset')
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print('Running locally — outputs will be saved to:', DRIVE_OUT)

print('DRIVE_OUT:', DRIVE_OUT)

In [ ]:
# === SETUP 4 — preprocess raw → JSONL (Dataset/Preprocessed/) ===
!python -m src.data.preprocess

In [ ]:
# === SETUP 5 — connect to local Ollama (gemma4:e4b), dùng chung cho 12 experiment ===
# Ollama must be running: `ollama serve` (and `ollama pull gemma4:e4b` if not yet downloaded)
from src.models.ollama import OllamaChatLM, OllamaConfig
from src.runner import load_config, run
from pathlib import Path

MODEL = OllamaChatLM(OllamaConfig(model_name='gemma4:e4b'))
MODEL.load()
print('Model ready:', MODEL.config.model_name)

def run_exp(cfg_path, *, verbose=True, verbose_first_n=5, verbose_every=200,
            running_score_every=100, output_dir=DRIVE_OUT,
            n_hypotheses=None, max_correction_attempts=None,
            inference_batch_size=None):
    cfg = load_config(cfg_path)
    cfg.output_dir = output_dir
    cfg.verbose = verbose
    cfg.verbose_first_n = verbose_first_n
    cfg.verbose_every = verbose_every
    cfg.running_score_every = running_score_every
    if n_hypotheses is not None:
        cfg.n_hypotheses = n_hypotheses
    if max_correction_attempts is not None:
        cfg.max_correction_attempts = max_correction_attempts
    if inference_batch_size is not None:
        cfg.inference_batch_size = inference_batch_size
    result = run(cfg, model=MODEL)

    out_dir = Path(output_dir) / cfg.method / cfg.dataset
    pred_file   = out_dir / 'predictions.jsonl'
    metric_file = out_dir / 'metrics.json'
    n_saved = sum(1 for _ in pred_file.open(encoding='utf-8')) if pred_file.exists() else 0
    print(f'\n── Saved outputs ──────────────────────────────')
    print(f'  predictions ({n_saved} rows): {pred_file}')
    print(f'  metrics              : {metric_file}')
    print(f'  summary              : {Path(output_dir) / "summary.csv"}')
    print(f'───────────────────────────────────────────────\n')
    return result

## 12 experiments — mỗi experiment 1 cell riêng

Các cell dưới đây độc lập: rerun cell nào chỉ ảnh hưởng kết quả của experiment đó. Output của mỗi experiment lưu tại `DRIVE_OUT/<method>/<dataset>/` (predictions.jsonl + metrics.json) và append 1 dòng vào `DRIVE_OUT/summary.csv`.

### Zero-shot

In [ ]:
# === EXP 1/8 — zero_shot × UDST-DurationQA (EN, Duration, F1) ===
m = run_exp('configs/zero_shot_udst_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 2/8 — zero_shot × BigBench_DateUnderstanding (EN, DateArith, Accuracy) ===
m = run_exp('configs/zero_shot_bigbench_date.yaml', verbose=True, verbose_every=50,
            running_score_every=50)
print(m['metrics'])

In [ ]:
# === EXP 3/8 — zero_shot × VLSP ViTempQA DateArith (VI, DateArith, Accuracy) ===
m = run_exp('configs/zero_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 4/8 — zero_shot × VLSP ViTempQA DurationQA (VI, Duration, F1) ===
m = run_exp('configs/zero_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

### Few-shot (k cố định, shots thủ công trong `src/prompts/shot_pools.py`)

In [ ]:
# === EXP 5/8 — few_shot k=4 × UDST-DurationQA (EN, Duration, F1) ===
m = run_exp('configs/few_shot_udst_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 6/8 — few_shot k=3 × BigBench_DateUnderstanding (EN, DateArith, Accuracy) ===
m = run_exp('configs/few_shot_bigbench_date.yaml', verbose=True, verbose_every=50,
            running_score_every=50)
print(m['metrics'])

In [ ]:
# === EXP 7/8 — few_shot k=3 × VLSP ViTempQA DateArith (VI, DateArith, Accuracy) ===
m = run_exp('configs/few_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 8/8 — few_shot k=4 × VLSP ViTempQA DurationQA (VI, Duration, F1) ===
m = run_exp('configs/few_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

### Symbolic CoT (program synthesis + symbolic execution)

LLM đóng vai **planner/compiler**: sinh Python datetime program → `temporal_executor` chạy deterministic → verify kết quả → self-correct nếu execution fail.

Tuỳ chỉnh:
- `n_hypotheses`: số chương trình độc lập mỗi sample (1 = nhanh, 3 = robust hơn).
- `max_correction_attempts`: số lần self-correction loop nếu program fail (0 = không sửa).

Các tham số này có thể override trong `run_exp()` hoặc chỉnh trực tiếp trong YAML config.

In [ ]:
# === EXP 9/12 — symbolic_cot × UDST-DurationQA (EN, Duration, F1) ===
m = run_exp('configs/symbolic_cot_udst_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 10/12 — symbolic_cot × BigBench_DateUnderstanding (EN, DateArith, Accuracy) ===
m = run_exp('configs/symbolic_cot_bigbench_date.yaml', verbose=True, verbose_every=50,
            running_score_every=50)
print(m['metrics'])

In [ ]:
# === EXP 11/12 — symbolic_cot × VLSP ViTempQA DateArith (VI, DateArith, Accuracy) ===
m = run_exp('configs/symbolic_cot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

In [ ]:
# === EXP 12/12 — symbolic_cot × VLSP ViTempQA DurationQA (VI, Duration, F1) ===
m = run_exp('configs/symbolic_cot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

## Debug helpers

Các cell dưới để audit chất lượng extractor / xem sample thất bại — dùng khi cần thôi.

In [ ]:
# === DEBUG A — load lại predictions của 1 run, xem các sample sai đầu tiên ===
import json
from pathlib import Path

METHOD = 'zero_shot'        # đổi theo experiment cần audit
DATASET = 'vlsp_date'       # đổi theo experiment cần audit
N_SHOW = 10

path = Path(DRIVE_OUT) / METHOD / DATASET / 'predictions.jsonl'
wrong = []
parse_fail = []
with open(path, encoding='utf-8') as f:
    for line in f:
        r = json.loads(line)
        if r['extracted'] is None:
            parse_fail.append(r)
        elif not r['correct']:
            wrong.append(r)

print(f'parse_fail={len(parse_fail)} wrong={len(wrong)}')
print('\n--- Sample parse failures ---')
for r in parse_fail[:N_SHOW]:
    print(f"  gold={r['gold_normalized']!r} raw={r['raw_output']!r}")
print('\n--- Sample wrong (but parsed) ---')
for r in wrong[:N_SHOW]:
    print(f"  gold={r['gold_normalized']!r} pred={r['extracted']!r} raw={r['raw_output'][:120]!r}")

In [ ]:
# === DEBUG B — thử prompt 1 sample bất kỳ cho nhanh (không qua runner) ===
from src.data.registry import load_dataset
from src.prompts.templates import build_messages
from src.prompts.shot_pools import get_shots
from src.evaluation.extractor import extract, normalize_gold

DATASET = 'vlsp_date'       # đổi dataset để probe
IDX = 0
K_SHOT = 0                  # 0 = zero-shot

samples = load_dataset(DATASET, max_samples=IDX + 1)
s = samples[IDX]
shots = get_shots(s['task'], s['language'], K_SHOT) if K_SHOT else ()
msgs = build_messages(s, shots=shots)
print('--- MESSAGES ---')
for m in msgs:
    print(f"[{m.role}] {m.content}")

raw = MODEL.generate(msgs, max_new_tokens=32, do_sample=False, enable_thinking=False)
print('\n--- RAW OUTPUT ---')
print(repr(raw))
print('\n--- EXTRACTED vs GOLD ---')
print('extracted:', extract(s['task'], s['language'], raw))
print('gold     :', normalize_gold(s['task'], s['language'], s['gold']))

In [ ]:
# === DEBUG C — xem bảng tổng quan summary.csv ===
import pandas as pd, os
df = pd.read_csv(os.path.join(DRIVE_OUT, 'summary.csv'))
df

### DEBUG D — Symbolic CoT message & execution trace (no GPU needed)

Trace toàn bộ message pipeline của `SymbolicCoTMethod` với **sample data tĩnh + canned LLM responses** — không cần load model.

| Trace | Task | Lang | Layers exercised |
|-------|------|------|-----------------|
| A | date\_arith | EN | L0 rule-based → L1 planner → L2A synthesis → L3 exec → L5a format verify → L5b retro verify |
| B | duration | EN | L0 rule-based → L1 planner → L2B CoT+KB |
| C | date\_arith | VI | L0 rule-based (Vietnamese path) |

Khi thấy `⚠ Rule-based returned None` → đó là sample cần LLM; messages bên dưới chính xác là những gì model sẽ nhận.  
Khi thấy `✓ Solved immediately` → Layer 0 đã xử lý xong, không gọi LLM.

In [ ]:
# === DEBUG D — Symbolic CoT pipeline trace (no GPU / model needed) ===
# Trace toàn bộ message flow qua 6 layer với sample data tĩnh + canned LLM responses.
# Chạy độc lập, không cần MODEL đã load ở SETUP 5.

import sys, textwrap

try:
    _repo = REPO_DIR  # set by SETUP 2
except NameError:
    import os; _repo = os.getcwd()
if _repo not in sys.path:
    sys.path.insert(0, _repo)

from src.utils.temporal_executor import (
    execute_program, verify_answer, extract_code_block, clean_code,
)
from src.utils.temporal_extractor import (
    solve_date_arith, solve_duration, _match_activity,
)
from src.methods.symbolic_cot import (
    _PLANNER_SYSTEM, _GUIDED_SYNTHESIS_SYSTEM,
    _DURATION_COT_SYSTEM, _VERIFIER_SYSTEM,
    _extract_yes_no, _extract_reasoning,
)
from src.models.base import ChatMessage

# ── ANSI colour helpers ───────────────────────────────────────────────────────
B = "\033[1m"; C = "\033[36m"; G = "\033[32m"; Y = "\033[33m"; R = "\033[31m"; X = "\033[0m"

def hdr(t):
    print(f"\n{B}{C}{'═'*72}{X}\n{B}{C}  {t}{X}\n{B}{C}{'═'*72}{X}")
def sub(t):
    print(f"\n{B}{Y}── {t} ──{X}")
def ok(m):   print(f"{G}✓ {m}{X}")
def bad(m):  print(f"{R}✗ {m}{X}")
def warn(m): print(f"{Y}⚠ {m}{X}")

def show_msgs(msgs):
    for m in msgs:
        body = textwrap.indent(m.content.strip(), "    ")
        print(f"  {B}[{m.role.upper()}]{X}\n{body}\n")

def show_canned(label, text):
    print(f"  {B}[MOCK LLM → {label}]{X}")
    print(textwrap.indent(text.strip(), "    "))

# ── Static sample data ────────────────────────────────────────────────────────
S_DATE_EN = {
    "task": "date_arith", "language": "en",
    "question": "Today is March 14, 2023. What is the date 100 days from now in MM/DD/YYYY?",
    "context": "", "gold": "06/22/2023", "meta": {},
}
S_DUR_EN = {
    "task": "duration", "language": "en",
    "question": "Is 2 hours a reasonable time to watch a movie?",
    "context": "A typical Hollywood feature film runs 90–150 minutes.",
    "gold": "yes", "meta": {"candidate_answer": "2 hours"},
}
S_DATE_VI = {
    "task": "date_arith", "language": "vi",
    "question": "Hôm nay là tháng 1, 2023. Sau 3 tháng là tháng mấy?",
    "context": "", "gold": "Tháng 4, 2023", "meta": {},
}

# ── Canned LLM responses (stand-in for real model output) ────────────────────
PLAN_EN = """\
1. Identify the start date: March 14, 2023.
2. Add 100 days using timedelta.
3. Format the result as MM/DD/YYYY."""

SYNTH_EN = """\
Reasoning:
1. Start date is March 14, 2023 → date(2023, 3, 14).
2. Add timedelta(days=100) to reach the target date.

Code:
```python
start = date(2023, 3, 14)
result = start + timedelta(days=100)
answer = result.strftime("%m/%d/%Y")
```"""

DUR_COT_EN = """\
Reasoning:
1. The event type is watching a movie.
2. Typical movie runtime: 90–150 minutes (1.5–2.5 h).
3. Candidate duration: 2 hours = 120 minutes — within the typical range.
4. Conclusion: yes, 2 hours is reasonable.
Answer: yes"""

VERIFY_RESP = "VALID"

# ═════════════════════════════════════════════════════════════════════════════
# TRACE A — date_arith (EN): full 6-layer pipeline
# ═════════════════════════════════════════════════════════════════════════════
s = S_DATE_EN
hdr(f"TRACE A  date_arith [EN]\n  Q: {s['question']}")

# Layer 0: Rule-based fast path
sub("L0 · Rule-based fast path (temporal_extractor)")
rule = solve_date_arith(s)
if rule is not None:
    ok(f"Solved immediately → {rule!r}   gold={s['gold']!r}   correct={rule == s['gold']}")
else:
    warn("Rule-based returned None → continues to LLM layers")

# Layer 1: Planner
sub("L1 · Planner  (CoT decomposition — 0.0 temp, 150 tokens)")
lang = s["language"]
q    = s["question"]
ctx  = (s.get("context") or "").strip()
planner_msgs = [
    ChatMessage(role="system", content=_PLANNER_SYSTEM.get(lang, _PLANNER_SYSTEM["en"])),
    ChatMessage(role="user",   content=f"Context: {ctx}\nQuestion: {q}" if ctx else f"Question: {q}"),
]
show_msgs(planner_msgs)
show_canned("plan", PLAN_EN)

# Layer 2A: Guided Synthesis (CoT reasoning + code, single LLM call)
sub("L2A · Guided Synthesis  (CoT steps + Python code, 300 tokens)")
synth_sys = _GUIDED_SYNTHESIS_SYSTEM.get((s["task"], lang),
                                          _GUIDED_SYNTHESIS_SYSTEM[("date_arith", "en")])
synth_msgs = [
    ChatMessage(role="system", content=synth_sys),
    ChatMessage(role="user",   content="\n".join([f"Question: {q}", f"Plan:\n{PLAN_EN}"])),
]
show_msgs(synth_msgs)
show_canned("synthesis", SYNTH_EN)

# Layer 3: Code extraction + Symbolic execution
sub("L3 · Symbolic execution  (extract_code_block → clean_code → exec)")
raw_code = extract_code_block(SYNTH_EN)
clean    = clean_code(raw_code)
print(f"  Extracted + cleaned:\n{textwrap.indent(clean, '    ')}")
answer, error = execute_program(SYNTH_EN, s["task"], lang)
if error:
    bad(f"execute_program error: {error}")
else:
    ok(f"execute_program → answer={answer!r}")

# Layer 5a: Format + calendar constraint check
sub("L5a · verify_answer  (format + calendar constraint)")
fmt_ok = verify_answer(answer, s["task"], lang)
(ok if fmt_ok else bad)(f"verify_answer({answer!r}, {s['task']!r}, {lang!r}) → {fmt_ok}")

# Layer 5b: Retrospective verifier (mock)
sub("L5b · Retrospective verifier  (faithfulness check, 50 tokens)")
reasoning = _extract_reasoning(SYNTH_EN)
verifier_msgs = [
    ChatMessage(role="system", content=_VERIFIER_SYSTEM),
    ChatMessage(role="user",   content=(
        f"Question: {q}\n\nReasoning:\n{reasoning}\n\n"
        f"Final Answer: {answer}\n\n"
        "Is this reasoning faithful and consistent with the answer?"
    )),
]
show_msgs(verifier_msgs)
show_canned("verifier", VERIFY_RESP)
retro_ok = "VALID" in VERIFY_RESP and "INVALID" not in VERIFY_RESP
(ok if retro_ok else warn)(f"Retrospective verify → {'PASSED' if retro_ok else 'FAILED'}")

sub("Final result")
correct = (answer or "").strip() == s["gold"].strip()
print(f"  Predicted : {answer!r}")
print(f"  Gold      : {s['gold']!r}")
(ok if correct else bad)(f"Correct: {correct}")


# ═════════════════════════════════════════════════════════════════════════════
# TRACE B — duration (EN): L0 → L1 → L2B (KB hint + CoT)
# ═════════════════════════════════════════════════════════════════════════════
s2 = S_DUR_EN
hdr(f"TRACE B  duration [EN]\n  Q: {s2['question']}")

sub("L0 · Rule-based fast path (temporal_extractor)")
rule2 = solve_duration(s2)
if rule2 is not None:
    ok(f"Solved immediately → {rule2!r}   gold={s2['gold']!r}   correct={rule2 == s2['gold']}")
else:
    warn("Rule-based returned None → continues to LLM layers")

sub("L1 · Planner  (CoT decomposition)")
lang2 = s2["language"]
ctx2  = s2.get("context") or ""
p_msgs2 = [
    ChatMessage(role="system", content=_PLANNER_SYSTEM.get(lang2, _PLANNER_SYSTEM["en"])),
    ChatMessage(role="user",   content=f"Context: {ctx2}\nQuestion: {s2['question']}"),
]
show_msgs(p_msgs2)

sub("L2B · Duration CoT + Knowledge-Base hint  (250 tokens)")
act_range = _match_activity(ctx2, s2["question"])
if act_range:
    lo, hi = act_range
    kb_hint = f"Knowledge base hint: typical duration for this event is {lo/60:.0f} min – {hi/3600:.1f} h."
    ok(f"_match_activity → {lo/60:.0f} min – {hi/3600:.1f} h")
else:
    kb_hint = ""
    warn("_match_activity: no keyword matched")

parts2 = [
    f"Context: {ctx2}",
    f"Question: {s2['question']}",
    f"Candidate duration: {s2['meta']['candidate_answer']}",
]
if kb_hint:
    parts2.append(kb_hint)
d_msgs = [
    ChatMessage(role="system", content=_DURATION_COT_SYSTEM.get(lang2, _DURATION_COT_SYSTEM["en"])),
    ChatMessage(role="user",   content="\n".join(parts2)),
]
show_msgs(d_msgs)
show_canned("duration CoT", DUR_COT_EN)

answer2 = _extract_yes_no(DUR_COT_EN)
ok(f"_extract_yes_no → {answer2!r}")

sub("Final result")
correct2 = answer2 == s2["gold"].strip().lower()
print(f"  Predicted : {answer2!r}")
print(f"  Gold      : {s2['gold']!r}")
(ok if correct2 else bad)(f"Correct: {correct2}")


# ═════════════════════════════════════════════════════════════════════════════
# TRACE C — date_arith (VI): L0 rule-based only
# ═════════════════════════════════════════════════════════════════════════════
s3 = S_DATE_VI
hdr(f"TRACE C  date_arith [VI]\n  Q: {s3['question']}")
sub("L0 · Rule-based fast path (temporal_extractor — Vietnamese path)")
rule3 = solve_date_arith(s3)
if rule3 is not None:
    ok(f"Solved immediately → {rule3!r}   gold={s3['gold']!r}   correct={rule3 == s3['gold']}")
else:
    warn("Rule-based returned None → LLM layers would run (see TRACE A for message templates)")


### DEBUG E — Symbolic CoT trace with REAL LLM output (requires MODEL from SETUP 5)

Giống DEBUG D nhưng dùng **MODEL thật** — mỗi layer gọi `MODEL.generate()` và in raw output.  
Dùng để kiểm tra: model có follow format không, plan có coherent không, code có chạy được không.

In [ ]:
# === DEBUG E — Symbolic CoT trace with REAL LLM output (requires MODEL from SETUP 5) ===
import textwrap

from src.utils.temporal_executor import execute_program, verify_answer, extract_code_block, clean_code
from src.utils.temporal_extractor import solve_date_arith, solve_duration, _match_activity
from src.methods.symbolic_cot import (
    _PLANNER_SYSTEM, _GUIDED_SYNTHESIS_SYSTEM,
    _DURATION_COT_SYSTEM, _VERIFIER_SYSTEM,
    _extract_yes_no, _extract_reasoning,
)
from src.models.base import ChatMessage

B="\033[1m"; C="\033[36m"; G="\033[32m"; Y="\033[33m"; R="\033[31m"; X="\033[0m"
def hdr(t):  print(f"\n{B}{C}{'═'*72}{X}\n{B}{C}  {t}{X}\n{B}{C}{'═'*72}{X}")
def sub(t):  print(f"\n{B}{Y}── {t} ──{X}")
def ok(m):   print(f"{G}✓ {m}{X}")
def bad(m):  print(f"{R}✗ {m}{X}")
def warn(m): print(f"{Y}⚠ {m}{X}")
def show_msgs(msgs):
    for m in msgs:
        print(f"  {B}[{m.role.upper()}]{X}\n{textwrap.indent(m.content.strip(), '    ')}\n")
def show_llm(label, text):
    print(f"  {B}[LLM → {label}]{X}")
    print(textwrap.indent((text or "").strip(), "    "))

# ── Sample data ───────────────────────────────────────────────────────────────
S_DATE_EN = {
    "task": "date_arith", "language": "en",
    "question": "Today is March 14, 2023. What is the date 100 days from now in MM/DD/YYYY?",
    "context": "", "gold": "06/22/2023", "meta": {},
}
S_DUR_EN = {
    "task": "duration", "language": "en",
    "question": "Is 2 hours a reasonable time to watch a movie?",
    "context": "A typical Hollywood feature film runs 90–150 minutes.",
    "gold": "yes", "meta": {"candidate_answer": "2 hours"},
}

# ═════════════════════════════════════════════════════════════════════════════
# TRACE A (LIVE) — date_arith EN: full pipeline with real LLM calls
# ═════════════════════════════════════════════════════════════════════════════
s = S_DATE_EN
hdr(f"TRACE A (LIVE)  date_arith [EN]\n  Q: {s['question']}")
lang = s["language"]
q    = s["question"]
ctx  = (s.get("context") or "").strip()

# L0: Rule-based
sub("L0 · Rule-based fast path")
rule = solve_date_arith(s)
if rule is not None:
    ok(f"Solved immediately → {rule!r}   (gold={s['gold']!r}   correct={rule == s['gold']})")
    print("  (Running LLM layers anyway for inspection)")
else:
    warn("Rule-based returned None → LLM required")

# L1: Planner
sub("L1 · Planner  (temp=0.0, max_new_tokens=150)")
planner_msgs = [
    ChatMessage(role="system", content=_PLANNER_SYSTEM.get(lang, _PLANNER_SYSTEM["en"])),
    ChatMessage(role="user",   content=f"Question: {q}" if not ctx else f"Context: {ctx}\nQuestion: {q}"),
]
show_msgs(planner_msgs)
plan_raw = MODEL.generate(planner_msgs, max_new_tokens=150, temperature=0.0,
                          do_sample=False, enable_thinking=False)
show_llm("plan", plan_raw)
plan = plan_raw.strip() or None

# L2A: Guided Synthesis
sub("L2A · Guided Synthesis  (temp=0.0, max_new_tokens=300)")
synth_sys = _GUIDED_SYNTHESIS_SYSTEM.get((s["task"], lang),
                                          _GUIDED_SYNTHESIS_SYSTEM[("date_arith", "en")])
synth_user = "\n".join([f"Question: {q}"] + ([f"Plan:\n{plan}"] if plan else []))
synth_msgs = [
    ChatMessage(role="system", content=synth_sys),
    ChatMessage(role="user",   content=synth_user),
]
show_msgs(synth_msgs)
synth_raw = MODEL.generate(synth_msgs, max_new_tokens=300, temperature=0.0,
                           do_sample=False, enable_thinking=False)
show_llm("synthesis", synth_raw)

# L3: Extract + Execute
sub("L3 · Symbolic execution  (extract_code_block → clean_code → exec)")
code_clean = clean_code(extract_code_block(synth_raw))
print(f"  Extracted + cleaned:\n{textwrap.indent(code_clean, '    ')}")
answer, error = execute_program(synth_raw, s["task"], lang)
if error:
    bad(f"execute_program error: {error}")
    # L4: Self-correction attempt
    sub("L4 · Self-correction  (temp=0.0, max_new_tokens=200)")
    from src.methods.symbolic_cot import _CORRECTION_SYSTEM
    corr_msgs = [
        ChatMessage(role="system", content=_CORRECTION_SYSTEM),
        ChatMessage(role="user",   content=(
            f"Error: {error}\n\nBuggy program:\n{code_clean}\n\n"
            f"Question: {q}\n\nWrite the corrected program:"
        )),
    ]
    show_msgs(corr_msgs)
    corr_raw = MODEL.generate(corr_msgs, max_new_tokens=200, temperature=0.0,
                              do_sample=False, enable_thinking=False)
    show_llm("correction", corr_raw)
    answer, error2 = execute_program(corr_raw, s["task"], lang)
    if error2:
        bad(f"Correction also failed: {error2}")
    else:
        ok(f"Corrected → answer={answer!r}")
else:
    ok(f"execute_program → answer={answer!r}")

# L5a: Format verify
sub("L5a · verify_answer")
fmt_ok = verify_answer(answer, s["task"], lang)
(ok if fmt_ok else bad)(f"verify_answer({answer!r}) → {fmt_ok}")

# L5b: Retrospective verifier
sub("L5b · Retrospective verifier  (temp=0.0, max_new_tokens=50)")
reasoning = _extract_reasoning(synth_raw) or synth_raw[:400]
verifier_msgs = [
    ChatMessage(role="system", content=_VERIFIER_SYSTEM),
    ChatMessage(role="user",   content=(
        f"Question: {q}\n\nReasoning:\n{reasoning}\n\n"
        f"Final Answer: {answer}\n\n"
        "Is this reasoning faithful and consistent with the answer?"
    )),
]
show_msgs(verifier_msgs)
verify_raw = MODEL.generate(verifier_msgs, max_new_tokens=50, temperature=0.0,
                            do_sample=False, enable_thinking=False)
show_llm("verifier", verify_raw)
retro_ok = "VALID" in verify_raw.upper() and "INVALID" not in verify_raw.upper()
(ok if retro_ok else warn)(f"Retrospective verify → {'PASSED' if retro_ok else 'FAILED'}")

sub("Final result")
correct = (answer or "").strip() == s["gold"].strip()
print(f"  Predicted : {answer!r}")
print(f"  Gold      : {s['gold']!r}")
(ok if correct else bad)(f"Correct: {correct}")


# ═════════════════════════════════════════════════════════════════════════════
# TRACE B (LIVE) — duration EN: L0 → L1 → L2B with KB hint
# ═════════════════════════════════════════════════════════════════════════════
s2 = S_DUR_EN
hdr(f"TRACE B (LIVE)  duration [EN]\n  Q: {s2['question']}")
lang2 = s2["language"]
ctx2  = s2.get("context") or ""

sub("L0 · Rule-based fast path")
rule2 = solve_duration(s2)
if rule2 is not None:
    ok(f"Solved immediately → {rule2!r}   (gold={s2['gold']!r}   correct={rule2 == s2['gold']})")
    print("  (Running LLM layers anyway for inspection)")
else:
    warn("Rule-based returned None → LLM required")

sub("L1 · Planner  (temp=0.0, max_new_tokens=150)")
p_msgs2 = [
    ChatMessage(role="system", content=_PLANNER_SYSTEM.get(lang2, _PLANNER_SYSTEM["en"])),
    ChatMessage(role="user",   content=f"Context: {ctx2}\nQuestion: {s2['question']}"),
]
show_msgs(p_msgs2)
plan2_raw = MODEL.generate(p_msgs2, max_new_tokens=150, temperature=0.0,
                           do_sample=False, enable_thinking=False)
show_llm("plan", plan2_raw)

sub("L2B · Duration CoT + KB hint  (temp=0.0, max_new_tokens=250)")
act_range = _match_activity(ctx2, s2["question"])
kb_hint = ""
if act_range:
    lo, hi = act_range
    kb_hint = f"Knowledge base hint: typical duration for this event is {lo/60:.0f} min – {hi/3600:.1f} h."
    ok(f"_match_activity → {lo/60:.0f} min – {hi/3600:.1f} h  (injected into prompt)")
else:
    warn("_match_activity: no keyword matched — no KB hint")

d_parts = [
    f"Context: {ctx2}",
    f"Question: {s2['question']}",
    f"Candidate duration: {s2['meta']['candidate_answer']}",
]
if kb_hint:
    d_parts.append(kb_hint)
d_msgs = [
    ChatMessage(role="system", content=_DURATION_COT_SYSTEM.get(lang2, _DURATION_COT_SYSTEM["en"])),
    ChatMessage(role="user",   content="\n".join(d_parts)),
]
show_msgs(d_msgs)
dur_raw = MODEL.generate(d_msgs, max_new_tokens=250, temperature=0.0,
                         do_sample=False, enable_thinking=False)
show_llm("duration CoT", dur_raw)

answer2 = _extract_yes_no(dur_raw)
(ok if answer2 else warn)(f"_extract_yes_no → {answer2!r}")

sub("Final result")
correct2 = answer2 == s2["gold"].strip().lower()
print(f"  Predicted : {answer2!r}")
print(f"  Gold      : {s2['gold']!r}")
(ok if correct2 else bad)(f"Correct: {correct2}")
